In [31]:
!pip install -q -U transformers accelerate huggingface_hub pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 118.7 MB/s eta 0:00:00


In [46]:
!pip install -q fastapi uvicorn pyngrok python-multipart

In [32]:
from huggingface_hub import login

login()

In [36]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "google/medgemma-1.5-4b-it"

processor = AutoProcessor.from_pretrained(
    model_id,
    token=True
)

vlm_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=True
)

print("MedGemma chargé avec succès")
print("Device :", vlm_model.device)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

MedGemma chargé avec succès
Device : cuda:0


In [37]:
from google.colab import files

uploaded = files.upload()

Saving IM-0001-0001.jpeg to IM-0001-0001 (1).jpeg


In [41]:
from google.colab import files

uploaded = files.upload()

Saving IM-0001-0001_gradcam.png to IM-0001-0001_gradcam (1).png


In [42]:
from google.colab import files

uploaded = files.upload()

Saving person1_virus_6.jpeg to person1_virus_6 (1).jpeg


In [43]:
from google.colab import files

uploaded = files.upload()

Saving person1_virus_6_gradcam.png to person1_virus_6_gradcam (1).png


In [39]:
# ============================================================
# FONCTION VLM GÉNÉRIQUE — VERSION FINALE
# ============================================================

from PIL import Image
import torch


def interpret_with_medgemma(
    xray_path,
    gradcam_path,
    predicted_class,
    pneumonia_probability,
    processor,
    vlm_model
):

    # 1. Charger les images
    xray = Image.open(xray_path).convert("RGB")
    gradcam = Image.open(gradcam_path).convert("RGB")

    # 2. Probabilité de la classe prédite
    if predicted_class == "PNEUMONIA":
        class_probability = pneumonia_probability
        decision_context = """
The CNN predicted PNEUMONIA.
Describe whether the spatial attention is anatomically plausible
for supporting this CNN prediction.
Do NOT state that highlighted regions necessarily represent pneumonia.
"""
    else:
        class_probability = 1.0 - pneumonia_probability
        decision_context = """
The CNN predicted NORMAL.
Interpret the highlighted regions only as areas that contributed
to the CNN's NORMAL decision.
Do NOT describe highlighted regions as abnormalities, lesions,
or suspicious findings solely because they are activated.
"""

    # 3. Prompt
    prompt = f"""
The first image is the original chest X-ray.
The second image is a Grad-CAM overlay generated by a DenseNet121 classifier.

CNN prediction: {predicted_class}
P(PNEUMONIA) = {pneumonia_probability:.6f}
P({predicted_class}) = {class_probability:.6f}

{decision_context}

Your task is ONLY to explain the CNN visual attention.
You are not being asked to independently diagnose the X-ray.

Important constraints:
- Grad-CAM was generated from a 7x7 feature map.
- Therefore, spatial localization is coarse.
- Do not claim precise anatomical localization unless clearly supported.
- Do not infer pathology simply because a region is highlighted.
- Do not override or reinterpret the CNN classification.
- Do not provide a definitive clinical diagnosis.
- Separate what is directly visible in the Grad-CAM from interpretation.
- If localization is uncertain, explicitly state the uncertainty.
- Use cautious terms such as "appears", "suggests", and "may".

Report exactly these six points:

1. Main regions emphasized by the Grad-CAM.

2. Lung-field localization:
   Is activation mainly inside the lung fields, partly outside them,
   or uncertain?

3. Spatial distribution:
   Is the activation focal, diffuse, unilateral, bilateral,
   or too coarse to determine reliably?

4. CNN-decision consistency:
   Describe whether the attention pattern appears anatomically
   plausible for supporting the CNN's {predicted_class} decision.
   Do not infer a new diagnosis.

5. Potential irrelevant attention:
   Is there visible evidence that the CNN may rely on image borders,
   text, shoulders, abdomen, diaphragm, or other non-lung structures?

6. Final Grad-CAM assessment:
   Briefly summarize the quality of the explanation and its main
   uncertainty. Do not repeat the entire previous analysis.
"""

    # 4. Message multimodal
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": xray},
                {"type": "image", "image": gradcam},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    # 5. Préparation
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(vlm_model.device)
        if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    # 6. Génération
    with torch.inference_mode():
        output = vlm_model.generate(
            **inputs,
            max_new_tokens=450,
            do_sample=False
        )

    generated_tokens = output[0][
        inputs["input_ids"].shape[-1]:
    ]

    interpretation = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return interpretation

In [47]:
# ============================================================
# API MEDGEMMA — FASTAPI
# ============================================================

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from PIL import Image
import tempfile
import os

app = FastAPI()


@app.get("/")
def home():
    return {
        "status": "ok",
        "message": "MedGemma API is running"
    }


@app.post("/interpret")
async def interpret(
    xray: UploadFile = File(...),
    gradcam: UploadFile = File(...),
    predicted_class: str = Form(...),
    pneumonia_probability: float = Form(...)
):

    xray_path = None
    gradcam_path = None

    try:

        # Sauvegarde temporaire de la radio
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".png"
        ) as temp_xray:

            temp_xray.write(
                await xray.read()
            )

            xray_path = temp_xray.name

        # Sauvegarde temporaire de la Grad-CAM
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".png"
        ) as temp_gradcam:

            temp_gradcam.write(
                await gradcam.read()
            )

            gradcam_path = temp_gradcam.name

        # Appel de MedGemma
        interpretation = interpret_with_medgemma(
            xray_path=xray_path,
            gradcam_path=gradcam_path,
            predicted_class=predicted_class,
            pneumonia_probability=pneumonia_probability,
            processor=processor,
            vlm_model=vlm_model
        )

        return JSONResponse(
            {
                "predicted_class": predicted_class,
                "pneumonia_probability": pneumonia_probability,
                "interpretation": interpretation
            }
        )

    except Exception as e:

        return JSONResponse(
            status_code=500,
            content={
                "error": str(e)
            }
        )

    finally:

        if xray_path and os.path.exists(xray_path):
            os.remove(xray_path)

        if gradcam_path and os.path.exists(gradcam_path):
            os.remove(gradcam_path)

In [44]:
interpretation = interpret_with_medgemma(
    xray_path="IM-0001-0001.jpeg",
    gradcam_path="IM-0001-0001_gradcam.png",
    predicted_class="NORMAL",
    pneumonia_probability=0.0378575436770916,
    processor=processor,
    vlm_model=vlm_model
)

print("===== INTERPRÉTATION MEDGEMMA =====\n")
print(interpretation)

===== INTERPRÉTATION MEDGEMMA =====

1. Main regions emphasized by the Grad-CAM: The Grad-CAM highlights the right lung field, specifically the lower portion.
2. Lung-field localization: The activation is mainly inside the lung fields.
3. Spatial distribution: The activation is focal and appears to be concentrated in the right lower lung field.
4. CNN-decision consistency: The attention pattern appears anatomically plausible for supporting the CNN's NORMAL decision, as it focuses on the lung tissue.
5. Potential irrelevant attention: There is no visible evidence of attention to non-lung structures.
6. Final Grad-CAM assessment: The explanation is based on the Grad-CAM overlay and provides a summary of the attention pattern. The main uncertainty is the precise localization due to the coarse nature of the Grad-CAM.


In [45]:
interpretation = interpret_with_medgemma(
    xray_path="person1_virus_6.jpeg",
    gradcam_path="person1_virus_6_gradcam.png",
    predicted_class="PNEUMONIA",
    pneumonia_probability=0.9997476935386658,
    processor=processor,
    vlm_model=vlm_model
)

print("===== INTERPRÉTATION MEDGEMMA =====\n")
print(interpretation)

===== INTERPRÉTATION MEDGEMMA =====

1. Main regions emphasized by the Grad-CAM: The Grad-CAM highlights the right lung field, particularly the upper and mid zones.

2. Lung-field localization: The activation is mainly inside the lung fields, with some extension into the right upper mediastinum.

3. Spatial distribution: The activation is diffuse and bilateral, with a greater emphasis on the right lung.

4. CNN-decision consistency: The attention pattern appears anatomically plausible for supporting the CNN's PNEUMONIA decision. The highlighted regions are within the lung parenchyma.

5. Potential irrelevant attention: There is some activation in the right upper mediastinum, which could be related to the endotracheal tube.

6. Final Grad-CAM assessment: The explanation is clear and addresses the key aspects of the Grad-CAM. The main uncertainty is the precise localization of the attention within the lung fields.


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("you'r token")

In [51]:
public_url = ngrok.connect(8000)

base_url = public_url.public_url

print("API publique :", base_url)
print("Endpoint MedGemma :", f"{base_url}/interpret")

API publique : https://stifle-deploy-emoticon.ngrok-free.dev
Endpoint MedGemma : https://stifle-deploy-emoticon.ngrok-free.dev/interpret
